# Exploración CLR y PCA: taxonomía y funciones

Este cuaderno tiene dos secciones de datos: abundancia taxonómica por género y
rutas funcionales. Cada una sólo hace la preparación específica de su formato.
Después ambas llaman a **la misma función** `run_clr_pca`, que filtra,
transforma a CLR, calcula PCA, genera el biplot y guarda las salidas finales.


In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

DATASETS = PROJECT_ROOT / "datasets"
FIGURES = PROJECT_ROOT / "results" / "figures" / "02_clr_pca_exploration"
TABLES = PROJECT_ROOT / "results" / "tables" / "02_clr_pca_exploration"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

METADATA = pd.read_excel(DATASETS / "metadata_LATINBIOTA_MEXICO.xlsx", sheet_name="Data")
OUTLIERS = [
    "37082_3#16", "37082_2#4", "37035_2#22",
    "37035_7#18", "36703_3#20", "36703_3#4",
]


## Función común: filtrar, CLR, PCA y biplot


In [6]:
def run_clr_pca(abundance, name, title, prevalence=0.05, top_features=8):
    # Aplica el mismo análisis CLR/PCA a una matriz muestras × características.
    # 1. Conservar muestras con etiqueta Rural o Urban y quitar outliers definidos.
    X = abundance.drop(index=OUTLIERS, errors="ignore").apply(pd.to_numeric, errors="coerce").fillna(0)
    lifestyle = METADATA.set_index("Lane")["Lifestyle"].reindex(X.index)
    valid = lifestyle.isin(["Rural", "Urban"])
    X, lifestyle = X.loc[valid], lifestyle.loc[valid]

    # 2. Agrupar características muy raras en Other para evitar matrices enormes.
    minimum_samples = max(1, int(np.ceil(prevalence * len(X))))
    keep = (X > 0).sum(axis=0) >= minimum_samples
    rare = X.columns[~keep]
    if len(rare):
        X["Other"] = X.loc[:, rare].sum(axis=1)
        X = X.loc[:, X.columns[keep].tolist() + ["Other"]]

    # 3. Transformación CLR: pseudoconteo para ceros y centrado logarítmico por muestra.
    positive_values = X.where(X > 0).stack()
    if positive_values.empty:
        raise ValueError(f"{name}: no hay abundancias positivas para aplicar CLR.")
    pseudocount = positive_values.min() / 2
    log_X = np.log(X + pseudocount)
    X_clr = log_X.sub(log_X.mean(axis=1), axis=0)

    # 4. PCA sobre los datos CLR; no se vuelve a estandarizar.
    pca = PCA(n_components=2)
    scores = pd.DataFrame(pca.fit_transform(X_clr), index=X.index, columns=["PC1", "PC2"])
    loadings = pd.DataFrame(pca.components_.T, index=X.columns, columns=["PC1", "PC2"])
    loadings["contribution"] = np.hypot(loadings["PC1"], loadings["PC2"])
    loadings = loadings.sort_values("contribution", ascending=False)

    # 5. Un único formato de biplot para ambos tipos de datos.
    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    colors = {"Rural": "#2a9d8f", "Urban": "#e76f51"}
    for group in ["Rural", "Urban"]:
        points = scores.loc[lifestyle.eq(group)]
        ax.scatter(points.PC1, points.PC2, label=group, color=colors[group], s=42,
                   alpha=0.8, edgecolor="white", linewidth=0.5)

    selected = loadings.head(top_features)
    scale = scores.abs().to_numpy().max() / selected[["PC1", "PC2"]].abs().to_numpy().max() * 0.65
    for feature, row in selected.iterrows():
        x, y = row.PC1 * scale, row.PC2 * scale
        ax.annotate("", xy=(x, y), xytext=(0, 0),
                    arrowprops={"arrowstyle": "->", "color": "#7f1d1d", "lw": 1.3})
        ax.text(x * 1.08, y * 1.08, feature, fontsize=7, color="#7f1d1d",
                ha="center", va="center")

    ax.axhline(0, color="0.65", lw=0.7)
    ax.axvline(0, color="0.65", lw=0.7)
    ax.grid(alpha=0.2, linestyle="--")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.set_title(title)
    ax.legend(title="Lifestyle", frameon=False)
    fig.tight_layout()

    scores.assign(Lifestyle=lifestyle).to_csv(TABLES / f"{name}_pca_scores.csv")
    loadings.to_csv(TABLES / f"{name}_pca_loadings.csv")
    fig.savefig(FIGURES / f"{name}_clr_pca_biplot.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES / f"{name}_clr_pca_biplot.pdf", bbox_inches="tight")
    plt.show()

    print(f"{name}: {len(X)} muestras, {X.shape[1]} características, PC1+PC2 = {pca.explained_variance_ratio_[:2].sum():.1%}")
    return scores.assign(Lifestyle=lifestyle), loadings


## 1. Taxonomía: géneros de MetaPhlAn


In [7]:
# Preparación específica: la tabla MetaPhlAn está en taxones × muestras.
metaphlan = pd.read_csv(DATASETS / "latinbiota_merge_metaphlan_data.csv", sep="\t", skiprows=1, index_col=0)
taxon_names = metaphlan.index.astype(str)
is_genus = (
    taxon_names.str.contains(r"(?:^|\|)g__", regex=True)
    & ~taxon_names.str.contains(r"(?:^|\|)[ts]__", regex=True)
    & ~taxon_names.str.contains("unclassified", case=False)
)
genus = metaphlan.loc[is_genus].copy()
genus.index = [name.split("|g__", 1)[1].split("|", 1)[0].replace("_", " ") for name in genus.index]
genus = genus.groupby(level=0).sum().T  # muestras × géneros

genus_scores, genus_loadings = run_clr_pca(
    genus,
    name="taxonomic_genus",
    title="PCA con CLR: composición taxonómica por género",
)
genus_loadings.head(10)


IndexError: boolean index did not match indexed array along axis 0; size of axis is 1263 but size of corresponding boolean axis is 1262

![Biplot taxonómico final](../results/figures/02_clr_pca_exploration/taxonomic_genus_biplot_final.png)


## 2. Funcional: rutas no estratificadas de HUMAnN


In [8]:
# Preparación específica: remover filas no asignadas y transponer rutas × muestras.
functional = pd.read_csv(DATASETS / "latinbiota_pathabundance_unstratified_relab.tsv", sep="\t", index_col=0)
functional = functional.drop(index=["UNMAPPED", "UNINTEGRATED"], errors="ignore")
functional = functional.loc[:, ~functional.columns.str.contains(r"\.1", regex=True)]
functional = functional.T
functional.index = functional.index.str.replace("_paired_Abundance", "", regex=False)

functional_scores, functional_loadings = run_clr_pca(
    functional,
    name="functional_pathways",
    title="PCA con CLR: rutas funcionales",
)
functional_loadings.head(10)


IndexError: boolean index did not match indexed array along axis 0; size of axis is 533 but size of corresponding boolean axis is 532

![Biplot funcional final](../results/figures/02_clr_pca_exploration/functional_pathways_biplot_final.png)
